# Projeto Final – Processamento de Imagens
## Classificação de imagens com GLCM (Haralick) + classificadores clássicos

**Abordagem:** Clássica (extração de características + classificação)
**Descritor:** GLCM – Gray-Level Co-occurrence Matrix (descritor de textura, diferente de HOG, LBP e histograma de cor vistos em aula)
**Classificadores:** SVM, Random Forest e KNN (comparação)
**Métricas:** matriz de confusão, precisão, recall, F1-score e acurácia

**Pipeline:** carregar dataset → holdout 80/20 → extrair GLCM → normalizar → treinar → avaliar

## 1. Bibliotecas

In [ ]:
import os, shutil, random, time
import numpy as np
import cv2
import matplotlib.pyplot as plt
from tqdm import tqdm

from skimage.feature import graycomatrix, graycoprops

from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, ConfusionMatrixDisplay)

## 2. Configuração do dataset

**Antes de rodar**, ajuste os caminhos abaixo. Há dois casos:

**Caso 1 (Coffee Bean do Kaggle):** a pasta já tem `train/` e `test/`, cada uma com as subpastas das classes (Dark, Green, Light, Medium). Use `USAR_SPLIT_PRONTO = True`.

**Caso 2:** dataset sem divisão, só com subpastas de classes:

```
dataset/
 ├── classe_A/  (imagens)
 └── classe_B/  (imagens)
```

Se o dataset estiver no Google Drive, a célula abaixo monta o Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:

# Caso 1 
USAR_SPLIT_PRONTO = True
HOLDOUT_DIR = '/content/drive/MyDrive/archive'   # pasta que contem train/ e test/

# Caso 2
DATASET_DIR = '/content/drive/MyDrive/dataset'   
SPLIT_OUT_DIR = '/content/dataset_holdout'      

SPLIT_RATIO = 0.8   
SEED = 42
IMG_SIZE = 224    

## 3. Divisão treino/teste (holdout 80/20)
Adaptado do exercício de holdout da aula (agora limpa a pasta de saída antes, para evitar duplicar arquivos ao reexecutar).

In [ ]:
def split_dataset(input_dataset_path, output_dataset_path, split_ratio=0.8, seed=42):
    if os.path.exists(output_dataset_path):
        shutil.rmtree(output_dataset_path)          
    os.makedirs(output_dataset_path)

    classes = sorted(d for d in os.listdir(input_dataset_path)
                     if os.path.isdir(os.path.join(input_dataset_path, d)))

    for class_name in classes:
        class_path = os.path.join(input_dataset_path, class_name)
        files = sorted(f for f in os.listdir(class_path)
                       if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')))

        random.seed(seed)
        random.shuffle(files)
        split_index = int(split_ratio * len(files))

        train_path = os.path.join(output_dataset_path, 'train', class_name)
        test_path  = os.path.join(output_dataset_path, 'test', class_name)
        os.makedirs(train_path, exist_ok=True)
        os.makedirs(test_path, exist_ok=True)

        for i, f in enumerate(files):
            dst = train_path if i < split_index else test_path
            shutil.copy(os.path.join(class_path, f), os.path.join(dst, f))

        print(f"Classe '{class_name}': {split_index} treino | {len(files) - split_index} teste")

if USAR_SPLIT_PRONTO:
    print('Usando a divisao train/test que ja vem no dataset:', HOLDOUT_DIR)
else:
    split_dataset(DATASET_DIR, SPLIT_OUT_DIR, SPLIT_RATIO, SEED)
    HOLDOUT_DIR = SPLIT_OUT_DIR

## 4. Carregamento das imagens
Cada imagem é lida, redimensionada e convertida para escala de cinza (o GLCM trabalha com níveis de cinza).

In [ ]:
def carregar_imagens(pasta):
    imagens, rotulos = [], []
    classes = sorted(os.listdir(pasta))
    for idx, classe in enumerate(classes):
        caminho = os.path.join(pasta, classe)
        for arq in tqdm(sorted(os.listdir(caminho)), desc=f'{os.path.basename(pasta)}/{classe}'):
            img = cv2.imread(os.path.join(caminho, arq), cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            imagens.append(img)
            rotulos.append(idx)
    return np.array(imagens), np.array(rotulos), classes

X_train_img, y_train, classes = carregar_imagens(os.path.join(HOLDOUT_DIR, 'train'))
X_test_img,  y_test,  _       = carregar_imagens(os.path.join(HOLDOUT_DIR, 'test'))

print('Classes:', classes)
print('Treino:', X_train_img.shape, '| Teste:', X_test_img.shape)

In [ ]:
# Exemplos 
fig, axs = plt.subplots(1, len(classes), figsize=(4*len(classes), 4))
axs = np.atleast_1d(axs)
for i, c in enumerate(classes):
    idx = np.where(y_train == i)[0][0]
    axs[i].imshow(X_train_img[idx], cmap='gray'); axs[i].set_title(c); axs[i].axis('off')
plt.show()

## 5. Extração de características – GLCM (Haralick)

A GLCM conta com que frequência pares de pixels com certos níveis de cinza aparecem lado a lado, em uma dada **distância** e **ângulo**. Dela extraímos 6 propriedades de textura:
`contrast`, `dissimilarity`, `homogeneity`, `energy`, `correlation`, `ASM`.

Parâmetros: 3 distâncias × 4 ângulos × 6 propriedades = **72 características por imagem**.
Os níveis de cinza são reduzidos de 256 para 64 para a matriz ficar menos esparsa e o cálculo mais rápido.

In [ ]:
DISTANCIAS = [1, 3, 5]
ANGULOS    = [0, np.pi/4, np.pi/2, 3*np.pi/4]
NIVEIS     = 64
PROPRIEDADES = ['contrast', 'dissimilarity', 'homogeneity', 'energy', 'correlation', 'ASM']

def extrair_glcm(imagens):
    lista = []
    for img in tqdm(imagens, desc='GLCM'):
        img_q = (img // (256 // NIVEIS)).astype(np.uint8)       # quantização para 64 níveis
        glcm = graycomatrix(img_q, DISTANCIAS, ANGULOS, levels=NIVEIS,
                            symmetric=True, normed=True)
        vetor = []
        for p in PROPRIEDADES:
            vetor.extend(graycoprops(glcm, p).ravel())
        lista.append(vetor)
    return np.array(lista)

t0 = time.time()
X_train = extrair_glcm(X_train_img)
X_test  = extrair_glcm(X_test_img)
print('Formato dos vetores:', X_train.shape, X_test.shape, f'| tempo: {time.time()-t0:.1f}s')

## 6. Normalização
O `StandardScaler` aprende média e desvio **somente no treino** e depois é aplicado no teste (evita vazamento de dados).

In [ ]:
scaler = StandardScaler()
X_train_n = scaler.fit_transform(X_train)
X_test_n  = scaler.transform(X_test)

## 7. Treino e comparação dos classificadores

In [ ]:
modelos = {
    'SVM (RBF)':     SVC(kernel='rbf', C=10, gamma='scale', random_state=SEED),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=SEED),
    'KNN (k=5)':     KNeighborsClassifier(n_neighbors=5),
}

resultados = {}
for nome, modelo in modelos.items():
    modelo.fit(X_train_n, y_train)
    pred = modelo.predict(X_test_n)
    resultados[nome] = (modelo, pred, accuracy_score(y_test, pred))

print(f"{'Classificador':<16} Acurácia")
for nome, (_, _, acc) in resultados.items():
    print(f"{nome:<16} {acc:.4f}")

## 8. Ajuste de hiperparâmetros do SVM (GridSearchCV)
Busca de `C` e `gamma` usando validação cruzada **apenas no treino**.

In [ ]:
grade = {'C': [0.1, 1, 10, 100], 'gamma': ['scale', 0.01, 0.1, 1]}
busca = GridSearchCV(SVC(kernel='rbf', random_state=SEED), grade, cv=5, n_jobs=-1)
busca.fit(X_train_n, y_train)

print('Melhores parâmetros:', busca.best_params_)
print(f'Acurácia média em validação cruzada: {busca.best_score_:.4f}')

svm_otimizado = busca.best_estimator_
pred_svm_otim = svm_otimizado.predict(X_test_n)
resultados['SVM otimizado'] = (svm_otimizado, pred_svm_otim, accuracy_score(y_test, pred_svm_otim))
print(f"Acurácia no teste (SVM otimizado): {resultados['SVM otimizado'][2]:.4f}")

## 9. Resultados finais (melhor modelo)
Matriz de confusão, precisão, recall, F1-score e acurácia.

In [ ]:
melhor_nome = max(resultados, key=lambda k: resultados[k][2])
melhor_modelo, y_pred, acc = resultados[melhor_nome]

print(f'Melhor modelo: {melhor_nome}')
print(f'Acurácia: {acc:.4f}\n')
print(classification_report(y_test, y_pred, target_names=classes, digits=4))

cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=classes).plot(cmap='Blues', xticks_rotation=45)
plt.title(f'Matriz de confusão – {melhor_nome}')
plt.tight_layout()
plt.show()

## 10. Comparação entre classificadores

In [ ]:
nomes = list(resultados.keys())
accs  = [resultados[n][2] for n in nomes]

plt.figure(figsize=(8, 4))
barras = plt.bar(nomes, accs, color='steelblue')
plt.ylim(0, 1.05); plt.ylabel('Acurácia'); plt.title('Acurácia por classificador (GLCM)')
for b, a in zip(barras, accs):
    plt.text(b.get_x() + b.get_width()/2, a + 0.01, f'{a:.3f}', ha='center')
plt.xticks(rotation=15); plt.tight_layout(); plt.show()

## 11. Conclusão

*(conclusão aqui)*